# OPF PII Benchmarks — Colab Runner

This notebook reproduces the [opf-benchmarks](https://github.com/CupOfGeo/opf-benchmarks-geo) suite: OpenAI Privacy Filter (OPF) evaluated against the three benchmarks NVIDIA published GLiNER-PII numbers on (Argilla PII, AI4Privacy, Nemotron-PII), in three modes each.

**Before running:**
1. Switch to a GPU runtime via *Runtime → Change runtime type → T4 GPU*. CPU works but takes much longer.
2. Add a HuggingFace token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar). Two of the three datasets (`ai4privacy/pii-masking-300k` and `nvidia/Nemotron-PII`) are gated — accept their terms on HuggingFace first, then create a read token at https://huggingface.co/settings/tokens.

Then *Runtime → Run all*. The headline F1 table renders in the last cell.

In [1]:
import os

REPO_DIR = "opf-benchmarks-geo"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/CupOfGeo/opf-benchmarks-geo.git
%cd {REPO_DIR}

Cloning into 'opf-benchmarks-geo'...
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 19 (delta 2), reused 19 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (19/19), 14.12 KiB | 2.82 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/Users/georgemazzeo/Code/opf-benchmarks/notebooks/opf-benchmarks-geo


In [2]:
!pip install -q .


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import os

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except (ImportError, Exception):
    token = os.environ.get("HF_TOKEN")

if not token:
    raise RuntimeError(
        "HF_TOKEN not found. Add it as a Colab secret (key icon in the left sidebar, "
        "name it HF_TOKEN, toggle 'Notebook access' on). Two of the three datasets "
        "are gated — accept terms at https://huggingface.co/datasets/ai4privacy/pii-masking-300k "
        "and https://huggingface.co/datasets/nvidia/Nemotron-PII first, then create a "
        "read token at https://huggingface.co/settings/tokens."
    )

os.environ["HF_TOKEN"] = token
print("HF_TOKEN loaded.")

HF_TOKEN loaded.


In [4]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: no GPU detected — runs will be slow. Switch to T4 via Runtime → Change runtime type.")

Using device: cpu


## Knobs

Tweak these to change the run. `MAX_EXAMPLES=1000` finishes in ~10–15 min on a free Colab T4 — bump to `5000` to match the sample size in the project README.

In [5]:
MAX_EXAMPLES = 1000
BENCHMARKS = ["argilla", "ai4privacy", "nemotron"]

In [6]:
!python -m scripts.download_datasets --max-examples {MAX_EXAMPLES} --benchmarks {" ".join(BENCHMARKS)}


=== argilla ===
  examples_full=1000 examples_opfscope=899 spans_in=1985 spans_out=1153

=== ai4privacy ===
  examples_full=1000 examples_opfscope=827 spans_in=4954 spans_out=831

=== nemotron ===
[nemotron] WARNING: 8 label(s) not in label_map; treated as out-of-scope: {'company_name': 529, 'time': 309, 'credit_debit_card': 137, 'employment_status': 97, 'date_time': 95, 'language': 62, 'http_cookie': 59, 'unique_id': 11}
  examples_full=1000 examples_opfscope=988 spans_in=5427 spans_out=2896
  unknown labels (added to OOS): {'credit_debit_card': 137, 'time': 309, 'http_cookie': 59, 'employment_status': 97, 'company_name': 529, 'date_time': 95, 'language': 62, 'unique_id': 11}


OSError: [Errno 5] Input/output error

In [ ]:
!python -m opf_benchmarks.run_eval --device {DEVICE} --benchmarks {" ".join(BENCHMARKS)}

In [ ]:
!python -m opf_benchmarks.aggregate

## Results

In [ ]:
from IPython.display import Markdown

Markdown(open("results/REPORT.md").read())

## (Optional) Download the report

Run the cell below in Colab to download `REPORT.md` to your machine. Skipped automatically when not running in Colab.

In [ ]:
try:
    from google.colab import files
    files.download("results/REPORT.md")
except ImportError:
    print("Not running in Colab — skipping download.")